# Module 4 — Learning Style Clustering + Study Plan Generator
### AI-Powered Student LMS | Phase 3

**Goal:** Group students by learning behaviour → generate personalized weekly study plans  
**Algorithm:** K-Means Clustering  
**Output:** 4 learning style clusters + auto-generated study plan per student  

| Cluster | Learning Style | Description |
|---|---|---|
| 0 | 🔴 Struggling Learner | Low grades, low approval rate, needs urgent help |
| 1 | 🟡 Average Performer | Mid-range, inconsistent, needs structure |
| 2 | 🟢 High Achiever | Excellent grades, high approval, self-directed |
| 3 | 🔵 Late Bloomer | Poor 1st sem, improved 2nd sem, needs motivation |

---

In [ ]:
# ── CELL 1 ── Imports & Paths ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib, os, json, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster          import KMeans
from sklearn.preprocessing    import StandardScaler
from sklearn.decomposition    import PCA
from sklearn.metrics          import silhouette_score

BASE   = r'C:\Users\DELL\Desktop\AI_Student_LMS'
MODELS = os.path.join(BASE, 'models')
ASSETS = os.path.join(BASE, 'assets')
DATA   = os.path.join(BASE, 'data', 'raw', 'data.csv')
PROC   = os.path.join(BASE, 'data', 'processed')

print('✅ All imports done!')

In [ ]:
# ── CELL 2 ── Load Data & Select Clustering Features ──────────
df = pd.read_csv(DATA, sep=';')
print(f'Dataset loaded: {df.shape}')

# Features that best describe learning behaviour/style
CLUSTER_FEATURES = [
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 2nd sem (evaluations)',
    'Admission grade',
    'Previous qualification (grade)',
    'Age at enrollment',
    'Scholarship holder',
    'Tuition fees up to date',
    'Debtor',
]

# Keep only existing columns
CLUSTER_FEATURES = [c for c in CLUSTER_FEATURES if c in df.columns]

X_cluster = df[CLUSTER_FEATURES].copy()
print(f'Clustering features: {len(CLUSTER_FEATURES)}')
print(CLUSTER_FEATURES)

In [ ]:
# ── CELL 3 ── Scale Features ───────────────────────────────────
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X_cluster)

print('✅ Features scaled!')
print(f'Scaled matrix shape: {X_scaled.shape}')

In [ ]:
# ── CELL 4 ── Elbow Method — Find Best K ──────────────────────
inertias    = []
sil_scores  = []
K_range     = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f'K={k} | Inertia={km.inertia_:.0f} | Silhouette={sil_scores[-1]:.4f}')

# Plot elbow + silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', markersize=8, linewidth=2)
axes[0].axvline(x=4, color='red', linestyle='--', label='Chosen K=4')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)')
axes[0].set_title('Elbow Method — Optimal K', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_xticks(list(K_range))

axes[1].plot(list(K_range), sil_scores, 'gs-', markersize=8, linewidth=2)
axes[1].axvline(x=4, color='red', linestyle='--', label='Chosen K=4')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].set_xticks(list(K_range))

plt.suptitle('Choosing Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'cluster_01_elbow.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Elbow plot saved!')

In [ ]:
# ── CELL 5 ── Train K-Means with K=4 ──────────────────────────
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10, max_iter=300)
df['cluster'] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['cluster'])
print(f'✅ K-Means trained with K={K}')
print(f'Silhouette Score: {sil:.4f}')
print(f'\nCluster sizes:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# ── CELL 6 ── Analyze & Label Each Cluster ────────────────────
# Compute mean of key features per cluster
key_cols = [
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Admission grade',
]
key_cols = [c for c in key_cols if c in df.columns]

cluster_summary = df.groupby('cluster')[key_cols].mean().round(2)
print('Cluster Feature Means:')
print(cluster_summary)

# Also show dropout rate per cluster
cluster_summary['dropout_rate_%'] = df.groupby('cluster').apply(
    lambda x: (x['Target'] == 'Dropout').sum() / len(x) * 100
).round(1).values

cluster_summary['size'] = df['cluster'].value_counts().sort_index().values
print('\nFull Cluster Summary (with dropout rate):')
print(cluster_summary)

# Auto-assign labels based on 2nd sem grade ranking
grade_col = 'Curricular units 2nd sem (grade)'
grade_rank = df.groupby('cluster')[grade_col].mean().rank()

CLUSTER_LABELS = {}
CLUSTER_COLORS = {}
for c in range(K):
    rank = grade_rank[c]
    if rank == 1:
        CLUSTER_LABELS[c] = '🔴 Struggling Learner'
        CLUSTER_COLORS[c] = '#E74C3C'
    elif rank == 2:
        CLUSTER_LABELS[c] = '🟡 Average Performer'
        CLUSTER_COLORS[c] = '#F39C12'
    elif rank == 3:
        CLUSTER_LABELS[c] = '🔵 Late Bloomer'
        CLUSTER_COLORS[c] = '#3498DB'
    else:
        CLUSTER_LABELS[c] = '🟢 High Achiever'
        CLUSTER_COLORS[c] = '#2ECC71'

df['learning_style'] = df['cluster'].map(CLUSTER_LABELS)

print('\nCluster → Learning Style mapping:')
for c, label in CLUSTER_LABELS.items():
    print(f'  Cluster {c} → {label}')

In [ ]:
# ── CELL 7 ── PLOT 1: PCA 2D Cluster Visualization ────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(11, 7))
for c in range(K):
    mask = df['cluster'] == c
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=CLUSTER_COLORS[c], alpha=0.5, s=18,
        label=f'Cluster {c}: {CLUSTER_LABELS[c]}'
    )

# Plot centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            c='black', s=200, marker='X', zorder=5, label='Centroids')

plt.title('K-Means Clusters — PCA 2D View', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'cluster_02_pca.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ PCA cluster plot saved!')

In [ ]:
# ── CELL 8 ── PLOT 2: Cluster Profile Radar / Bar ─────────────
profile_cols = [
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Admission grade',
]
profile_cols = [c for c in profile_cols if c in df.columns]
short_names  = ['1st Grade', '2nd Grade', '1st Approved', '2nd Approved', 'Admission']

profile_df = df.groupby('cluster')[profile_cols].mean()
# Normalize 0-1 for radar
profile_norm = (profile_df - profile_df.min()) / (profile_df.max() - profile_df.min())

fig, axes = plt.subplots(1, K, figsize=(18, 5))
for c in range(K):
    vals  = profile_norm.loc[c].values
    color = CLUSTER_COLORS[c]
    axes[c].barh(short_names, vals, color=color, alpha=0.8, edgecolor='white')
    axes[c].set_xlim(0, 1)
    axes[c].set_title(CLUSTER_LABELS[c], fontsize=10, fontweight='bold')
    axes[c].set_xlabel('Score (normalized)')
    size = (df['cluster'] == c).sum()
    axes[c].text(0.5, -0.15, f'n = {size} students',
                 transform=axes[c].transAxes,
                 ha='center', fontsize=9, color='gray')

plt.suptitle('Cluster Learning Profiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'cluster_03_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Cluster profiles saved!')

In [ ]:
# ── CELL 9 ── PLOT 3: Dropout Rate per Cluster ────────────────
dropout_rates = df.groupby('learning_style').apply(
    lambda x: (x['Target'] == 'Dropout').sum() / len(x) * 100
).sort_values(ascending=False)

colors_bar = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']

plt.figure(figsize=(10, 5))
bars = plt.bar(range(len(dropout_rates)), dropout_rates.values,
               color=colors_bar, edgecolor='white', linewidth=1.5)
plt.xticks(range(len(dropout_rates)),
           [l.split(' ', 1)[1] for l in dropout_rates.index],
           fontsize=10)
plt.ylabel('Dropout Rate (%)')
plt.title('Dropout Rate by Learning Style Cluster', fontsize=13, fontweight='bold')
plt.axhline(df['Target'].eq('Dropout').mean()*100,
            color='navy', linestyle='--', label='Overall avg')
plt.legend()
for bar, val in zip(bars, dropout_rates.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'cluster_04_dropout_rate.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dropout rate by cluster saved!')

In [ ]:
# ── CELL 10 ── Study Plan Generator (Rule-Based Engine) ────────

STUDY_PLANS = {
    '🔴 Struggling Learner': {
        'description': 'Students with low grades and high dropout risk. Needs urgent structured support.',
        'weekly_hours': 30,
        'priority':     'Catch up on failed units, attend all classes',
        'schedule': {
            'Monday':    '3h — Revise weakest subject + solve 10 practice problems',
            'Tuesday':   '3h — Attend extra tutorial session + complete missed assignments',
            'Wednesday': '4h — Study group session + previous year exam practice',
            'Thursday':  '3h — Revise 2nd weakest subject + topic summary notes',
            'Friday':    '3h — Mock test (1 subject) + error analysis',
            'Saturday':  '4h — Full revision of week topics + teacher consultation',
            'Sunday':    '2h — Light reading + plan next week goals',
        },
        'resources':    ['Khan Academy', 'NPTEL lectures', 'Past exam papers'],
        'alert':        '⚠️ HIGH DROPOUT RISK — Immediate faculty intervention recommended',
    },
    '🟡 Average Performer': {
        'description': 'Students with average performance needing consistency and better study habits.',
        'weekly_hours': 22,
        'priority':     'Build consistency, improve weaker subjects',
        'schedule': {
            'Monday':    '3h — Core subject study + assignment completion',
            'Tuesday':   '2h — Review lecture notes + concept mapping',
            'Wednesday': '3h — Practice problems (medium difficulty)',
            'Thursday':  '2h — Collaborative study + doubt clearing',
            'Friday':    '3h — Subject revision + mini quiz',
            'Saturday':  '4h — Deep dive into weak topic + lab practice',
            'Sunday':    '1.5h — Weekly review + goal setting',
        },
        'resources':    ['Coursera', 'YouTube subject channels', 'Textbook exercises'],
        'alert':        '📊 MEDIUM RISK — Monitor progress bi-weekly',
    },
    '🔵 Late Bloomer': {
        'description': 'Students who started slow but showed improvement. Needs motivation and momentum.',
        'weekly_hours': 20,
        'priority':     'Maintain improvement momentum, build confidence',
        'schedule': {
            'Monday':    '2.5h — Strengthen recently improved topics',
            'Tuesday':   '2.5h — New topic study + notes',
            'Wednesday': '3h — Problem solving + peer teaching',
            'Thursday':  '2h — Self-assessment quiz + gap analysis',
            'Friday':    '3h — Project or practical work',
            'Saturday':  '3h — Review week + prepare for next unit',
            'Sunday':    '2h — Relaxed reading + motivation journaling',
        },
        'resources':    ['Podcasts', 'Visual learning platforms', 'Group projects'],
        'alert':        '🔵 LOW-MEDIUM RISK — Encourage and track trajectory',
    },
    '🟢 High Achiever': {
        'description': 'Top performers with excellent grades. Focus on depth, research, and advanced topics.',
        'weekly_hours': 15,
        'priority':     'Advanced topics, research, competitive preparation',
        'schedule': {
            'Monday':    '2h — Advanced topic / research paper reading',
            'Tuesday':   '2h — Competitive exam practice (GATE/GRE prep)',
            'Wednesday': '2h — Side project / open source contribution',
            'Thursday':  '2h — Peer mentoring / teaching juniors',
            'Friday':    '2h — Hackathon / challenge problems',
            'Saturday':  '3h — Deep research or portfolio building',
            'Sunday':    '1h — Reflect + plan innovation goals',
        },
        'resources':    ['Research papers (arXiv)', 'LeetCode', 'GitHub projects', 'Internship prep'],
        'alert':        '✅ LOW RISK — Encourage higher challenges',
    },
}

print('✅ Study plan templates defined for all 4 learning styles!')
for style, plan in STUDY_PLANS.items():
    print(f'  {style} — {plan["weekly_hours"]}h/week')

In [ ]:
# ── CELL 11 ── Generate Plan for Individual Student ────────────
def generate_study_plan(student_features: dict) -> dict:
    """
    Given a student's feature dict, predict their cluster
    and return a personalized study plan.
    """
    # Build feature vector in correct order
    vec = np.array([[student_features.get(f, 0) for f in CLUSTER_FEATURES]])
    vec_scaled = scaler_cluster.transform(vec)
    cluster_id = kmeans.predict(vec_scaled)[0]
    style      = CLUSTER_LABELS[cluster_id]
    plan       = STUDY_PLANS[style]

    return {
        'cluster_id':     int(cluster_id),
        'learning_style': style,
        'description':    plan['description'],
        'weekly_hours':   plan['weekly_hours'],
        'priority':       plan['priority'],
        'schedule':       plan['schedule'],
        'resources':      plan['resources'],
        'alert':          plan['alert'],
    }


# ── Demo: Generate plan for a sample student ──────────────────
sample_student = {
    'Curricular units 1st sem (grade)':      8.5,
    'Curricular units 2nd sem (grade)':      7.0,
    'Curricular units 1st sem (approved)':   3,
    'Curricular units 2nd sem (approved)':   2,
    'Curricular units 1st sem (enrolled)':   6,
    'Curricular units 2nd sem (enrolled)':   6,
    'Curricular units 1st sem (evaluations)':6,
    'Curricular units 2nd sem (evaluations)':5,
    'Admission grade':                       110.0,
    'Previous qualification (grade)':        120.0,
    'Age at enrollment':                     20,
    'Scholarship holder':                    0,
    'Tuition fees up to date':               1,
    'Debtor':                                0,
}

result = generate_study_plan(sample_student)

print('=' * 55)
print('PERSONALIZED STUDY PLAN — DEMO STUDENT')
print('=' * 55)
print(f"Learning Style : {result['learning_style']}")
print(f"Description    : {result['description']}")
print(f"Weekly Hours   : {result['weekly_hours']} hours")
print(f"Priority       : {result['priority']}")
print(f"Alert          : {result['alert']}")
print(f"\n📅 WEEKLY SCHEDULE:")
for day, task in result['schedule'].items():
    print(f"  {day:<12}: {task}")
print(f"\n📚 Recommended Resources:")
for r in result['resources']:
    print(f"  • {r}")

In [ ]:
# ── CELL 12 ── Generate Plans for ALL Students → Save CSV ─────
plans_list = []
for idx, row in df.iterrows():
    student_dict = {f: row[f] for f in CLUSTER_FEATURES if f in row.index}
    plan = generate_study_plan(student_dict)
    plans_list.append({
        'student_index': idx,
        'actual_target': row['Target'],
        'cluster_id':    plan['cluster_id'],
        'learning_style':plan['learning_style'],
        'weekly_hours':  plan['weekly_hours'],
        'priority':      plan['priority'],
        'alert':         plan['alert'],
    })

plans_df = pd.DataFrame(plans_list)
plans_df.to_csv(os.path.join(PROC, 'student_study_plans.csv'), index=False)

print(f'✅ Study plans generated for all {len(plans_df)} students!')
print(f'Saved → data/processed/student_study_plans.csv')
print(f'\nLearning style distribution:')
print(plans_df['learning_style'].value_counts())

In [ ]:
# ── CELL 13 ── Save Models ─────────────────────────────────────
joblib.dump(kmeans,          os.path.join(MODELS, 'kmeans.pkl'))
joblib.dump(scaler_cluster,  os.path.join(MODELS, 'cluster_scaler.pkl'))
joblib.dump(CLUSTER_FEATURES,os.path.join(MODELS, 'cluster_features.pkl'))
joblib.dump(CLUSTER_LABELS,  os.path.join(MODELS, 'cluster_labels.pkl'))
joblib.dump(STUDY_PLANS,     os.path.join(MODELS, 'study_plans.pkl'))

# Save generate_study_plan function info as JSON for Streamlit
with open(os.path.join(MODELS, 'study_plans.json'), 'w') as f:
    # Convert emoji keys for JSON
    json.dump(
        {str(k): v for k, v in STUDY_PLANS.items()},
        f, indent=2
    )

print('✅ All models saved:')
print('   models/kmeans.pkl')
print('   models/cluster_scaler.pkl')
print('   models/cluster_features.pkl')
print('   models/cluster_labels.pkl')
print('   models/study_plans.pkl')
print('   models/study_plans.json')

In [ ]:
# ── CELL 14 ── Final Summary ───────────────────────────────────
print('=' * 55)
print('MODULE 4 — STUDY PLAN GENERATOR COMPLETE ✅')
print('=' * 55)
print(f'Algorithm        : K-Means (K=4)')
print(f'Silhouette Score : {sil:.4f}')
print(f'Students covered : {len(df)}')
print(f'\nLearning Style Groups:')
for c in range(K):
    n = (df['cluster'] == c).sum()
    pct = n / len(df) * 100
    print(f'  {CLUSTER_LABELS[c]:<30} : {n} students ({pct:.1f}%)')

print(f'\n📁 Models saved to models/')
print(f'📊 Charts saved to assets/')
print(f'📄 Plans CSV → data/processed/student_study_plans.csv')
print(f'\n' + '='*55)
print('ALL 4 ML MODULES COMPLETE! 🎉')
print('='*55)
print('\nModules done:')
print('  ✅ Module 1 — Anomaly Detection   (Isolation Forest)')
print('  ✅ Module 2 — Dropout Classifier  (Random Forest + SHAP)')
print('  ✅ Module 3 — Score Predictor     (XGBoost Regression)')
print('  ✅ Module 4 — Study Plan          (K-Means Clustering)')
print('\n🚀 Next: Build Streamlit Dashboard!')